## Анализ структуры владения компаний

Цель работы — очистка, нормализация и аналитический разбор данных о владельцах компаний.

Искяндярова Фортуна Ильдаровна, 25.02.2026

#0. Загрузка и первичное знакомство с данными

На данном этапе производится загрузка исходного файла и первичный осмотр структуры данных для понимания дальнейшей работы.

In [ ]:
from google.colab import files
import pandas as pd

#загружаем из файлов для коллаба
uploaded = files.upload()

#читаем файл
df = pd.read_excel("../data/raw/corporate_links_raw.xlsx")

#смотрим первые строки
df.head()
df.info()

Saving corporate_links_raw.xlsx to corporate_links_raw (3).xlsx
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 1 columns):
 #   Column                                                                     Non-Null Count  Dtype 
---  ------                                                                     --------------  ----- 
 0   FIO_owner,Company_name,INN_company,Ownership,Region,Source,Ownership_date  13 non-null     object
dtypes: object(1)
memory usage: 236.0+ bytes


Мы видим что датасет загрузился однострочно, а не по колонкам, а это значит, что файл имеет расширение csv, а не внешний xlx. Помимо того мы заметили, что датасет состоит из одной колонки в которой построчно перечислаются данные.



In [ ]:
import pandas as pd
import csv
from io import StringIO

#читаем как Excel
raw = pd.read_excel("../data/raw/corporate_links_raw.xlsx", header=None)

#берём единственную колонку (в ней csv-строки)
lines = raw.iloc[:, 0].astype(str).tolist()

#парсим как CSV-текст (учитывая кавычки в ООО "Ромашка" и т.д)
text = "\n".join(lines)
reader = csv.reader(StringIO(text), delimiter=",", quotechar='"', skipinitialspace=True)

rows = list(reader)
header = rows[0]
data = rows[1:]

df = pd.DataFrame(data, columns=header)

df.head(13)

,FIO_owner,Company_name,INN_company,Ownership,Region,Source,Ownership_date
0,Иванов И.И.,"ООО ""Ромашка""",7701234567,50%,Москва,Kommersant,12.03.2021
1,Иванов Иван Иванович,ООО Ромашка,7701234567,0.5,Москва,Kommersant,2021-03-12
2,Сидоров С.С,"ООО ""Ромашка""",7701234567,50 %,Москва,,15/03/2021
3,Петров Петр Петрович,"ООО ""Лютик""",7801234567,100%,Санкт-Петербург,RBC,01-04-2020
4,Петров П.П.,"ООО ""Лютик""",7801234567,30%,СПб,Forbes,2022/05/10
5,Иванов И.И.,"ООО ""Лютик""",7801234567,30 %,Москва,Forbes,10.05.2022
6,Смирнов А.А.,"АО ""Вектор""",5401234567,25%,Новосибирск,Kommersant,2019-07-01
7,Смирнов Алексей Алексеевич,АО Вектор,5401234567,0.25,Новосибирск,,01.07.2019
8,Кузнецова Е.В.,"ООО ""Альфа""",,40%,Москва,RBC,05.06.2020
9,Орлов Д.Д.,"ООО ""Альфа""",7723456789,70%,Москва,Kommersant,2020-06-05


## 1. Очистка и нормализация данных

В рамках данного этапа выполняется приведение данных к единому формату в соответствии с требованиями задания.

In [ ]:
import pandas as pd
import numpy as np
import re


#ФИО -> Фамилия И.О.

def normalize_fio(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s == "":
        return np.nan
    s = re.sub(r"\s+", " ", s)

    #уже "Фамилия И.О." или "Фамилия И. О."
    m = re.match(r"^([А-ЯЁA-Z][а-яёa-z\-]+)\s+([А-ЯЁA-Z])\.\s*([А-ЯЁA-Z])\.?$", s)
    if m:
        return f"{m.group(1)} {m.group(2)}.{m.group(3)}."

    #полное ФИО или ФИ
    parts = s.split(" ")
    if len(parts) >= 2:
        last = parts[0]
        first = parts[1]
        patr = parts[2] if len(parts) >= 3 else ""

        fi = (first[0].upper() + ".") if first else ""
        pi = (patr[0].upper() + ".") if patr else ""

        #Фамилия И.О. (или Фамилия И.)
        return f"{last} {fi}{pi}".strip()

    return s

df["FIO_owner_norm"] = df["FIO_owner"].apply(normalize_fio)

#проверим корректно ли записалось
df[["FIO_owner", "FIO_owner_norm"]].head(13)

,FIO_owner,FIO_owner_norm
0,Иванов И.И.,Иванов И.И.
1,Иванов Иван Иванович,Иванов И.И.
2,Сидоров С.С,Сидоров С.С.
3,Петров Петр Петрович,Петров П.П.
4,Петров П.П.,Петров П.П.
5,Иванов И.И.,Иванов И.И.
6,Смирнов А.А.,Смирнов А.А.
7,Смирнов Алексей Алексеевич,Смирнов А.А.
8,Кузнецова Е.В.,Кузнецова Е.В.
9,Орлов Д.Д.,Орлов Д.Д.


In [ ]:
#перезапишем колонку после проверки
df["FIO_owner"] = df["FIO_owner_norm"]
df.drop(columns=["FIO_owner_norm"], inplace=True)
df.head(13)

,FIO_owner,Company_name,INN_company,Ownership,Region,Source,Ownership_date
0,Иванов И.И.,"ООО ""Ромашка""",7701234567,50%,Москва,Kommersant,12.03.2021
1,Иванов И.И.,ООО Ромашка,7701234567,0.5,Москва,Kommersant,2021-03-12
2,Сидоров С.С.,"ООО ""Ромашка""",7701234567,50 %,Москва,,15/03/2021
3,Петров П.П.,"ООО ""Лютик""",7801234567,100%,Санкт-Петербург,RBC,01-04-2020
4,Петров П.П.,"ООО ""Лютик""",7801234567,30%,СПб,Forbes,2022/05/10
5,Иванов И.И.,"ООО ""Лютик""",7801234567,30 %,Москва,Forbes,10.05.2022
6,Смирнов А.А.,"АО ""Вектор""",5401234567,25%,Новосибирск,Kommersant,2019-07-01
7,Смирнов А.А.,АО Вектор,5401234567,0.25,Новосибирск,,01.07.2019
8,Кузнецова Е.В.,"ООО ""Альфа""",,40%,Москва,RBC,05.06.2020
9,Орлов Д.Д.,"ООО ""Альфа""",7723456789,70%,Москва,Kommersant,2020-06-05


In [ ]:
#сохраним бэкап на всякий случай
df_backup = df.copy()
#Company_name -> без кавычек и лишних пробелов
def normalize_company(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s == "":
        return np.nan
    #разные типы кавычек
    s = s.replace('"', "").replace("«", "").replace("»", "")
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["Company_name_norm"] = df["Company_name"].apply(normalize_company)
df[["Company_name_norm", "Company_name"]].head(13)

,Company_name_norm,Company_name
0,ООО Ромашка,"ООО ""Ромашка"""
1,ООО Ромашка,ООО Ромашка
2,ООО Ромашка,"ООО ""Ромашка"""
3,ООО Лютик,"ООО ""Лютик"""
4,ООО Лютик,"ООО ""Лютик"""
5,ООО Лютик,"ООО ""Лютик"""
6,АО Вектор,"АО ""Вектор"""
7,АО Вектор,АО Вектор
8,ООО Альфа,"ООО ""Альфа"""
9,ООО Альфа,"ООО ""Альфа"""


In [ ]:
#перезаписываем т.к всё выглядит корректно
df["Company_name"] = df["Company_name_norm"]
df.drop(columns=["Company_name_norm"], inplace=True)
df.head()

,FIO_owner,Company_name,INN_company,Ownership,Region,Source,Ownership_date
0,Иванов И.И.,ООО Ромашка,7701234567,50%,Москва,Kommersant,12.03.2021
1,Иванов И.И.,ООО Ромашка,7701234567,0.5,Москва,Kommersant,2021-03-12
2,Сидоров С.С.,ООО Ромашка,7701234567,50 %,Москва,,15/03/2021
3,Петров П.П.,ООО Лютик,7801234567,100%,Санкт-Петербург,RBC,01-04-2020
4,Петров П.П.,ООО Лютик,7801234567,30%,СПб,Forbes,2022/05/10


In [ ]:
from datetime import datetime
import re
import pandas as pd

#ИНН -> строка (только цифры)
#+ флаг отсутствия
def normalize_inn(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "none"}:
        return np.nan

    #7701234567.0 -> 7701234567
    if s.endswith(".0"):
        s = s[:-2]

    s = re.sub(r"\D", "", s)  #только цифры
    return s if s != "" else np.nan

df["INN_company_str"] = df["INN_company"].apply(normalize_inn)
df["INN_missing_flag"] = df["INN_company_str"].isna()

df_inn_missing = df[df["INN_missing_flag"]].copy()  #"обозначить отдельно"

#Ownership -> проценты float
#0.5 -> 50
#"50%" / "50 %" -> 50
#пустые -> NaN + флаг
def normalize_ownership(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "none"}:
        return np.nan

    s = re.sub(r"\s+", "", s)        #убрать пробелы
    s = s.replace(",", ".")          #0,5 -> 0.5

    has_percent = "%" in s
    s_num = s.replace("%", "")

    #если вообще нет цифр
    if not re.search(r"\d", s_num):
        return np.nan

    try:
        val = float(s_num)
    except:
        return np.nan

    if has_percent:
        return val

    #если доля от 0 до 1 -> переводим в проценты
    if 0 <= val <= 1:
        return val * 100

    #иначе считаем, что это уже проценты
    return val

df["Ownership_pct"] = df["Ownership"].apply(normalize_ownership).astype(float)
df["Ownership_missing_flag"] = df["Ownership_pct"].isna()
df_ownership_missing = df[df["Ownership_missing_flag"]].copy()  #пустые зн-я выделить


#Даты -> YYYY-MM-DD

def normalize_date(x):
    if pd.isna(x):
        return pd.NaT

    s = str(x)

    #чистим невидимые пробелы и мусор
    s = s.replace("\xa0", " ").strip()
    if s == "":
        return pd.NaT
    s = re.sub(r"\s+", "", s)  #убрать все пробелы внутри

    formats = [
        "%d.%m.%Y",  # 12.03.2021
        "%Y-%m-%d",  # 2021-03-12
        "%d/%m/%Y",  # 15/03/2021
        "%d-%m-%Y",  # 01-04-2020
        "%Y/%m/%d",  # 2022/05/10
        "%Y.%m.%d",  # 2021.11.12
    ]

    for fmt in formats:
        try:
            return datetime.strptime(s, fmt)
        except ValueError:
            continue

    return pd.NaT

df["Ownership_date_parsed"] = df["Ownership_date"].apply(normalize_date)
df["Ownership_date_norm"] = df["Ownership_date_parsed"].dt.strftime("%Y-%m-%d")
df["Ownership_date_unparsed_flag"] = df["Ownership_date_parsed"].isna()

#Контроль качества
summary = pd.DataFrame({
    "rows_total": [len(df)],
    "inn_missing": [int(df["INN_missing_flag"].sum())],
    "ownership_missing": [int(df["Ownership_missing_flag"].sum())],
    "date_unparsed": [int(df["Ownership_date_unparsed_flag"].sum())],
})

summary


,rows_total,inn_missing,ownership_missing,date_unparsed
0,13,1,1,0


In [ ]:
df[["INN_company", "INN_company_str"]].head(13)

,INN_company,INN_company_str
0,7701234567,7701234567
1,7701234567,7701234567
2,7701234567,7701234567
3,7801234567,7801234567
4,7801234567,7801234567
5,7801234567,7801234567
6,5401234567,5401234567
7,5401234567,5401234567
8,,NaN
9,7723456789,7723456789


In [ ]:
df[["Ownership", "Ownership_pct"]].head(13)

,Ownership,Ownership_pct
0,50%,50.0
1,0.5,50.0
2,50 %,50.0
3,100%,100.0
4,30%,30.0
5,30 %,30.0
6,25%,25.0
7,0.25,25.0
8,40%,40.0
9,70%,70.0


In [ ]:
df[["Ownership_date", "Ownership_date_norm"]].head(13)

,Ownership_date,Ownership_date_norm
0,12.03.2021,2021-03-12
1,2021-03-12,2021-03-12
2,15/03/2021,2021-03-15
3,01-04-2020,2020-04-01
4,2022/05/10,2022-05-10
5,10.05.2022,2022-05-10
6,2019-07-01,2019-07-01
7,01.07.2019,2019-07-01
8,05.06.2020,2020-06-05
9,2020-06-05,2020-06-05


In [ ]:
#сохраним бэкап на всякий случай
df_backup = df.copy()
#INN
if "INN_company_str" in df.columns:
    df["INN_company"] = df["INN_company_str"]
    df.drop(columns=["INN_company_str"], inplace=True)
    print("INN_company обновлён и нормализован.")
else:
    print("INN_company_str не найден — вероятно, уже удалён.")

#Ownership
if "Ownership_pct" in df.columns:
    df["Ownership"] = df["Ownership_pct"]
    df.drop(columns=["Ownership_pct"], inplace=True)
    print("Ownership приведён к процентному формату (float).")
else:
    print("Ownership_pct не найден — вероятно, уже удалён.")

#Dates
if "Ownership_date_norm" in df.columns:
    df["Ownership_date"] = df["Ownership_date_norm"]
    print("Ownership_date приведена к формату YYYY-MM-DD.")

else:
    print("Ownership_date_norm не найден — вероятно, уже удалён.")

#удаляем вспомогательные колонки безопасно
to_drop = [c for c in ["Ownership_date_norm", "Ownership_date_parsed"] if c in df.columns]
if to_drop:
    df.drop(columns=to_drop, inplace=True)
    print("Вспомогательные колонки дат удалены.")

INN_company обновлён и нормализован.
Ownership приведён к процентному формату (float).
Ownership_date приведена к формату YYYY-MM-DD.
Вспомогательные колонки дат удалены.


In [ ]:

df.head(13)

,FIO_owner,Company_name,INN_company,Ownership,Region,Source,Ownership_date,INN_missing_flag,Ownership_missing_flag,Ownership_date_unparsed_flag
0,Иванов И.И.,ООО Ромашка,7701234567,50.0,Москва,Kommersant,2021-03-12,False,False,False
1,Иванов И.И.,ООО Ромашка,7701234567,50.0,Москва,Kommersant,2021-03-12,False,False,False
2,Сидоров С.С.,ООО Ромашка,7701234567,50.0,Москва,,2021-03-15,False,False,False
3,Петров П.П.,ООО Лютик,7801234567,100.0,Санкт-Петербург,RBC,2020-04-01,False,False,False
4,Петров П.П.,ООО Лютик,7801234567,30.0,СПб,Forbes,2022-05-10,False,False,False
5,Иванов И.И.,ООО Лютик,7801234567,30.0,Москва,Forbes,2022-05-10,False,False,False
6,Смирнов А.А.,АО Вектор,5401234567,25.0,Новосибирск,Kommersant,2019-07-01,False,False,False
7,Смирнов А.А.,АО Вектор,5401234567,25.0,Новосибирск,,2019-07-01,False,False,False
8,Кузнецова Е.В.,ООО Альфа,NaN,40.0,Москва,RBC,2020-06-05,True,False,False
9,Орлов Д.Д.,ООО Альфа,7723456789,70.0,Москва,Kommersant,2020-06-05,False,False,False


In [ ]:
#сводка по проблемам
pd.DataFrame({
    "rows_total": [len(df)],
    "inn_missing": [df["INN_missing_flag"].sum()],
    "ownership_missing": [df["Ownership_missing_flag"].sum()],
    "date_unparsed": [df["Ownership_date_unparsed_flag"].sum()],
})

,rows_total,inn_missing,ownership_missing,date_unparsed
0,13,1,1,0


In [ ]:
#пропущенные ИНН
df[df["INN_missing_flag"]][["FIO_owner","Company_name","INN_company","Ownership","Ownership_date","Source"]]

,FIO_owner,Company_name,INN_company,Ownership,Ownership_date,Source
8,Кузнецова Е.В.,ООО Альфа,NaN,40.0,2020-06-05,RBC


In [ ]:
#пропущенные %
df[df["Ownership_missing_flag"]][["FIO_owner","Company_name","Ownership","Ownership_date","Source"]]

,FIO_owner,Company_name,Ownership,Ownership_date,Source
11,Лебедев М.М.,ООО Гамма,NaN,2021-11-12,Kommersant


#1.1 Дополнительная очистка категориальных признаков

Помимо требований задания выполнена дополнительная унификация категориальных признаков (Region, Source) для обеспечения корректной агрегации данных на этапе анализа.

In [ ]:
#сохраним бэкап на всякий случай
df_backup = df.copy()

Пропуски в признаках Region и Source будут сохранены как NaN и не требуют отдельной маркировки в виде флага, поскольку не влияют на корректность ключевых расчётов (доли владения и идентификация компании).

In [ ]:
import numpy as np

#Region
df["Region"] = df["Region"].astype(str).str.strip()

#пустые строки в NaN
df["Region"] = df["Region"].replace("", np.nan)

#унификация
region_map = {
    "СПб": "Санкт-Петербург",
    "С.-Петербург": "Санкт-Петербург",
    "санкт-петербург": "Санкт-Петербург"
}

df["Region"] = df["Region"].replace(region_map)

#Source
df["Source"] = df["Source"].astype(str).str.strip()

#пустые строки -> NaN
df["Source"] = df["Source"].replace("", np.nan)

#если после strip осталась строка 'nan' — заменяем
df["Source"] = df["Source"].replace("Nan", np.nan)

In [ ]:
df["Region"].value_counts(dropna=False)

,count
Region,
Москва,7
Санкт-Петербург,2
Новосибирск,2
Екатеринбург,2


In [ ]:
df["Source"].value_counts(dropna=False)

,count
Source,
Kommersant,5
NaN,3
Forbes,3
RBC,2


In [ ]:
df.head(13)

,FIO_owner,Company_name,INN_company,Ownership,Region,Source,Ownership_date,INN_missing_flag,Ownership_missing_flag,Ownership_date_unparsed_flag
0,Иванов И.И.,ООО Ромашка,7701234567,50.0,Москва,Kommersant,2021-03-12,False,False,False
1,Иванов И.И.,ООО Ромашка,7701234567,50.0,Москва,Kommersant,2021-03-12,False,False,False
2,Сидоров С.С.,ООО Ромашка,7701234567,50.0,Москва,NaN,2021-03-15,False,False,False
3,Петров П.П.,ООО Лютик,7801234567,100.0,Санкт-Петербург,RBC,2020-04-01,False,False,False
4,Петров П.П.,ООО Лютик,7801234567,30.0,Санкт-Петербург,Forbes,2022-05-10,False,False,False
5,Иванов И.И.,ООО Лютик,7801234567,30.0,Москва,Forbes,2022-05-10,False,False,False
6,Смирнов А.А.,АО Вектор,5401234567,25.0,Новосибирск,Kommersant,2019-07-01,False,False,False
7,Смирнов А.А.,АО Вектор,5401234567,25.0,Новосибирск,NaN,2019-07-01,False,False,False
8,Кузнецова Е.В.,ООО Альфа,NaN,40.0,Москва,RBC,2020-06-05,True,False,False
9,Орлов Д.Д.,ООО Альфа,7723456789,70.0,Москва,Kommersant,2020-06-05,False,False,False


Выполним проверку дубликатов записей.

In [ ]:
df[df.duplicated(keep=False)]

,FIO_owner,Company_name,INN_company,Ownership,Region,Source,Ownership_date,INN_missing_flag,Ownership_missing_flag,Ownership_date_unparsed_flag
0,Иванов И.И.,ООО Ромашка,7701234567,50.0,Москва,Kommersant,2021-03-12,False,False,False
1,Иванов И.И.,ООО Ромашка,7701234567,50.0,Москва,Kommersant,2021-03-12,False,False,False


Обнаружены полные дублирующиеся записи.
Для предотвращения искажения агрегированных показателей выполнено удаление технических дублей.

In [ ]:
df = df.drop_duplicates()

In [ ]:
df = df.reset_index(drop=True) #сбросим индексы чтобы 1 не пропускался

In [ ]:
df.head(13)

,FIO_owner,Company_name,INN_company,Ownership,Region,Source,Ownership_date,INN_missing_flag,Ownership_missing_flag,Ownership_date_unparsed_flag
0,Иванов И.И.,ООО Ромашка,7701234567,50.0,Москва,Kommersant,2021-03-12,False,False,False
1,Сидоров С.С.,ООО Ромашка,7701234567,50.0,Москва,NaN,2021-03-15,False,False,False
2,Петров П.П.,ООО Лютик,7801234567,100.0,Санкт-Петербург,RBC,2020-04-01,False,False,False
3,Петров П.П.,ООО Лютик,7801234567,30.0,Санкт-Петербург,Forbes,2022-05-10,False,False,False
4,Иванов И.И.,ООО Лютик,7801234567,30.0,Москва,Forbes,2022-05-10,False,False,False
5,Смирнов А.А.,АО Вектор,5401234567,25.0,Новосибирск,Kommersant,2019-07-01,False,False,False
6,Смирнов А.А.,АО Вектор,5401234567,25.0,Новосибирск,NaN,2019-07-01,False,False,False
7,Кузнецова Е.В.,ООО Альфа,NaN,40.0,Москва,RBC,2020-06-05,True,False,False
8,Орлов Д.Д.,ООО Альфа,7723456789,70.0,Москва,Kommersant,2020-06-05,False,False,False
9,Орлов Д.Д.,ООО Альфа,7723456789,35.0,Москва,Forbes,2021-08-01,False,False,False


In [ ]:
print(f"Итоговое количество записей: {df.shape[0]}")
print(f"Количество признаков: {df.shape[1]}")

Итоговое количество записей: 12
Количество признаков: 10


In [ ]:
#сохраняем очищенные данные с колонками флагов
df.to_csv("../data/interim/corporate_links_clean_with_flags.csv", index=False)

#и версию без флагов тоже сохраним
df_clean = df.drop(columns=[
    "INN_missing_flag",
    "Ownership_missing_flag",
    "Ownership_date_unparsed_flag"
])

df_clean.to_csv("../data/clean/corporate_links_clean.csv", index=False)

In [ ]:
print(f"Итоговое количество записей: {df_clean.shape[0]}")
print(f"Количество признаков: {df_clean.shape[1]}")

Итоговое количество записей: 12
Количество признаков: 7


#Итог очистки и нормализации данных

В ходе подготовки датасета выполнено:


*   Нормализация ФИО владельцев к формату «Фамилия И.О.»
*   Унификация наименований компаний (удалены кавычки и лишние пробелы)
* Приведение ИНН к строковому типу, выделены записи с отсутствующим ИНН
* Приведение долей владения к единому числовому формату (в процентах)
* Обработка и унификация форматов дат (YYYY-MM-DD), выделены некорректные значения
* Проверка и удаление технических дублей
* Дополнительная унификация категориальных признаков (Region)


В результате сформированы две версии датасета:

* Версия с контрольными флагами для проверки качества данных.
* Финальная clean-версия без служебных колонок для аналитической обработки.

#2. Анализ данных


Задача №1
* Найти компании, где суммарная доля владельцев > 100%

In [ ]:
#группируем по ИНН и по компании т.к названия могут совпадать, а инн -- уникален
ownership_sum = (
    df_clean
    .groupby(["Company_name", "INN_company"], dropna=False)["Ownership"]
    .sum()
    .reset_index()
)

ownership_sum_over_100 = ownership_sum[ownership_sum["Ownership"] > 100]

ownership_sum_over_100

,Company_name,INN_company,Ownership
1,ООО Альфа,7723456789,105.0
4,ООО Лютик,7801234567,160.0


Проведена агрегация долей по компаниям.  
Дополнительно выполним проверку с учётом временного фактора.

In [ ]:
df_clean[
    df_clean["Company_name"].isin(["ООО Альфа", "ООО Лютик"])
].sort_values(["Company_name", "Ownership_date"])

,FIO_owner,Company_name,INN_company,Ownership,Region,Source,Ownership_date
7,Кузнецова Е.В.,ООО Альфа,NaN,40.0,Москва,RBC,2020-06-05
8,Орлов Д.Д.,ООО Альфа,7723456789,70.0,Москва,Kommersant,2020-06-05
9,Орлов Д.Д.,ООО Альфа,7723456789,35.0,Москва,Forbes,2021-08-01
2,Петров П.П.,ООО Лютик,7801234567,100.0,Санкт-Петербург,RBC,2020-04-01
3,Петров П.П.,ООО Лютик,7801234567,30.0,Санкт-Петербург,Forbes,2022-05-10
4,Иванов И.И.,ООО Лютик,7801234567,30.0,Москва,Forbes,2022-05-10


In [ ]:
ownership_sum_by_date = (
    df_clean
    .groupby(["Company_name", "INN_company", "Ownership_date"], dropna=False)["Ownership"]
    .sum()
    .reset_index()
)

ownership_sum_by_date[ownership_sum_by_date["Ownership"] > 100]

,Company_name,INN_company,Ownership_date,Ownership


#Вывод по задаче №1

При агрегировании долей владения без учёта временного фактора выявлены компании, где суммарная доля превышает 100%.

Однако при анализе структуры владения с учётом даты превышений в рамках одного периода не обнаружено.

Следовательно, превышение 100% связано с изменением состава владельцев во времени, а не с ошибками данных.

#Задача №2
Найти компании, где доли менялись во времени

Логика выполнения будет выглядеть так

Доля изменилась, если:

* один и тот же владелец

* в одной и той же компании

* в разные даты

* имеет разные значения Ownership

То есть проверяем уникальность долей во времени.

In [ ]:
ownership_changes = (
    df_clean
    .groupby(["Company_name", "INN_company", "FIO_owner"])["Ownership"]
    .nunique()
    .reset_index()
)

#оставляем тех, у кого больше 1 уникального значения
ownership_changed = ownership_changes[ownership_changes["Ownership"] > 1]

ownership_changed

,Company_name,INN_company,FIO_owner,Ownership
1,ООО Альфа,7723456789,Орлов Д.Д.,2
5,ООО Лютик,7801234567,Петров П.П.,2


In [ ]:
df_clean.merge(
    ownership_changed[["Company_name", "INN_company", "FIO_owner"]],
    on=["Company_name", "INN_company", "FIO_owner"],
    how="inner"
).sort_values(["Company_name", "FIO_owner", "Ownership_date"])

,FIO_owner,Company_name,INN_company,Ownership,Region,Source,Ownership_date
2,Орлов Д.Д.,ООО Альфа,7723456789,70.0,Москва,Kommersant,2020-06-05
3,Орлов Д.Д.,ООО Альфа,7723456789,35.0,Москва,Forbes,2021-08-01
0,Петров П.П.,ООО Лютик,7801234567,100.0,Санкт-Петербург,RBC,2020-04-01
1,Петров П.П.,ООО Лютик,7801234567,30.0,Санкт-Петербург,Forbes,2022-05-10


#Вывод по задаче №2

В двух компаниях зафиксировано изменение долей отдельных владельцев во времени:

ООО Альфа — Орлов Д.Д. (70% -> 35%)

ООО Лютик — Петров П.П. (100% -> 30%)

Это подтверждает динамический характер структуры собственности и необходимость анализа с учётом временного фактора.

#Задача №3
Найти владельцев, участвующих более чем в одной компании

Логика:

* группируем по FIO_owner

* считаем количество уникальных Company_name

* оставляем > 1

In [ ]:
owners_multiple_companies = (
    df_clean
    .groupby("FIO_owner")["Company_name"]
    .nunique()
    .reset_index()
)

owners_multiple_companies = owners_multiple_companies[
    owners_multiple_companies["Company_name"] > 1
]
owners_multiple_companies

,FIO_owner,Company_name
0,Иванов И.И.,3


In [ ]:
df_clean[df_clean["FIO_owner"] == "Иванов И.И."] \
    .sort_values("Company_name")

,FIO_owner,Company_name,INN_company,Ownership,Region,Source,Ownership_date
11,Иванов И.И.,ООО Гамма,6601234567,60.0,Екатеринбург,NaN,2021-11-12
4,Иванов И.И.,ООО Лютик,7801234567,30.0,Москва,Forbes,2022-05-10
0,Иванов И.И.,ООО Ромашка,7701234567,50.0,Москва,Kommersant,2021-03-12


#Вывод по задаче №3

Выявлен владелец, участвующий более чем в одной компании:

Иванов И.И. — 3 компании

Это может свидетельствовать о диверсификации бизнеса или участии в нескольких корпоративных структурах.

#Краткое описание результатов анализа
1. **Какие данные вызывают сомнение?**

В ходе анализа были выявлены следующие потенциально проблемные аспекты данных:

* Превышение суммарной доли владения >100%

При агрегировании без учёта временного фактора для некоторых компаний суммарная доля владельцев превышала 100%.
Однако дополнительная проверка по датам показала, что превышение связано с изменением структуры владения во времени, а не с одновременным распределением долей.

Тем не менее, подобная ситуация может вводить в заблуждение при некорректной агрегации данных.

* Отсутствие ИНН

Выявлены записи с отсутствующим ИНН компании.
Поскольку ИНН является ключевым идентификатором юридического лица, его отсутствие снижает точность сопоставления данных и может приводить к ошибкам при объединении или группировке.

* Пропуски в долях владения

Обнаружены записи с отсутствующим значением Ownership.
Отсутствие доли владения делает невозможным корректный расчёт структуры собственности.

* Разные источники и пропуски источников

Часть записей не содержит информации об источнике (Source).
Это затрудняет проверку достоверности данных и их происхождения.

2. **Где возможны ошибки источников?**

Потенциальные источники ошибок могут включать:

* Ошибки при ручном сборе данных

Если информация агрегировалась вручную из СМИ, возможны:
опечатки в процентах,
дублирование публикаций,
некорректное указание даты.

* Разные публикации о разных периодах

СМИ могут публиковать данные о структуре собственности в разные моменты времени.
При объединении таких данных без учёта даты возникает иллюзия превышения 100%.

* Неполные публикации

Некоторые источники могут указывать только одного владельца без полного состава собственников, что искажает картину распределения долей.

* Отсутствие стандартизации

Различные форматы дат, написания ФИО и названий компаний указывают на отсутствие унифицированного источника данных.

**Итог**

После очистки и нормализации датасет приведён к структурированному виду, пригодному для аналитической обработки.
Ключевые аномалии объясняются динамикой структуры владения, а не критическими ошибками данных.